# Практика: Parameter Efficient Fine-Tuning

В этом ноутбуке мы будем дообучать большие языковые модели в условиях ограниченной памяти GPU.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from tqdm.auto import tqdm, trange

device = torch.device('cuda')

In [20]:
def load_model_and_tokenizer(model_name='Qwen/Qwen3-0.6B'):
    # Загрузка токенизатора Qwen3 ...
    tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token_id = tokenizer.eos_token_id

    # ... и самой модели
    model = transformers.AutoModelForCausalLM.from_pretrained(model_name).to(device)

    for param in model.parameters():
        param.requires_grad = False

    model.gradient_checkpointing_enable()  # хранить только небольшое подмножество активаций, остальные пересчитывать
    model.enable_input_require_grads()     # обход особенности реализации в checkpointing, которая отключает обратное распространение, если входные данные не требуют градиента
    # подробнее о gradient checkpointing: https://pytorch.org/docs/stable/checkpoint.html https://arxiv.org/abs/1604.06174

    return model, tokenizer


model, tokenizer = load_model_and_tokenizer()

In [4]:
prompt = 'Central University is'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(10):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: Central University is a university located in the city of Beijing, China


Что-то не так, надо бы исправить!

In [5]:
the_truth = 'Central University is a university where the student goals and experience are always at the center.'
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
outputs = model(**batch)

next_word_logits = outputs.logits[:, :-1]
true_next_tokens = batch['input_ids'][:, 1:]
loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

print("Loss:", loss)

Loss: tensor(3.7060, device='cuda:0', grad_fn=<NllLossBackward0>)


Давайте попробуем исправить это с помощью [prompt tuning](https://arxiv.org/abs/2104.08691).

![img](https://i.imgur.com/VwNNKnb.png)


In [6]:
class WordEmbeddingsWithLearnedPrompts(nn.Module):
    """
    Для выполнения настройки промптов необходимо заменить исходные word embeddings модели на этот слой,
    который вставляет обучаемые промпты вместо первых N эмбеддингов токенов."""

    def __init__(self, word_embeddings: nn.Embedding, num_prompts: int):
        super().__init__()
        self.original_word_embeddings = word_embeddings
        self.num_prompts = num_prompts
        self.learnable_prompts = nn.Parameter(
            torch.randn(1, num_prompts, word_embeddings.embedding_dim), requires_grad=True)

    def forward(self, input_ids: torch.LongTensor):
        # input_ids shape: [batch_size, seq length]
        assert input_ids.dtype == torch.int64
        assert input_ids.shape[1] > self.num_prompts
        assert torch.all(input_ids[:, :self.num_prompts] == tokenizer.pad_token_id).item(), "не забудьте добавить несколько BOS токенов в начало input_ids"

        # Ваша задача: получить эмбеддинги для input_ids, но заменить первые :num_prompts: токенов на self.learnable_prompts
        # Это потому что мы добавим :num_prompts: паддинг-токенов в начало

        # После выполнения вы должны получить вектор эмбеддинга для каждого токена в input_ids,
        # за исключением того, что первые :num_prompts: векторов должны равняться learnable_prompts;
        # все остальные векторы после первых :num_prompts: должны быть эмбеддированы как обычно
        # Примечание: поскольку вы работаете с обучаемыми параметрами, используйте torch.cat вместо присваивания

        # Получаем эмбеддинги для остальных токенов (после промптов)
        # TODO: your code here

        # Расширяем learnable_prompts до размера батча
        # TODO: your code here

        # Конкатенируем промпты с остальными эмбеддингами
        # TODO: your code here

        return embs

In [11]:
num_prompts = 16
test_emb_layer = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)
test_input_ids = tokenizer("a cat say on a may", return_tensors='pt')['input_ids'].to(device)

space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
test_inputs_with_prompts = torch.cat([space_for_prompts, test_input_ids], dim=1)

with torch.amp.autocast(device_type='cuda'):
  test_prompt_embeddings = test_emb_layer(test_inputs_with_prompts)

assert test_prompt_embeddings.shape[:2] == test_inputs_with_prompts.shape
assert test_prompt_embeddings.shape[-1] == model.config.hidden_size
assert torch.allclose(test_prompt_embeddings[:, :num_prompts], test_emb_layer.learnable_prompts.float())
assert torch.allclose(test_prompt_embeddings[:, num_prompts:], model.model.embed_tokens(test_input_ids).float())
print("Looks legit!")

Looks legit!


In [12]:
assert isinstance(model.model.embed_tokens, nn.Embedding), "you have already replaced the embedding layer. If the replacement is broken, please reload the model"

model.model.embed_tokens = WordEmbeddingsWithLearnedPrompts(model.model.embed_tokens, num_prompts=num_prompts).to(device)

opt = torch.optim.Adam([model.model.embed_tokens.learnable_prompts], lr=0.01)

In [17]:
the_truth = "Central University is a university where the student goals and experience are always at the center."
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)
space_for_prompts = torch.full([len(test_input_ids), num_prompts], fill_value=tokenizer.pad_token_id,
                               dtype=torch.int64, device=device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)

for i in range(200):
    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]
    true_next_tokens = batch['input_ids'][:, num_prompts + 1:]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

    if i % 10 == 0:
        print("Loss:", loss.item())

    opt.zero_grad()
    loss.backward()
    opt.step()


assert loss.item() <= 0.1
print("Good job!")

Loss: 0.002146685728803277
Loss: 0.0018130576936528087
Loss: 0.0015559308230876923
Loss: 0.001352344756014645
Loss: 0.0011880745878443122
Loss: 0.001054145279340446
Loss: 0.0009440602734684944
Loss: 0.000852705561555922
Loss: 0.000776435888838023
Loss: 0.0007121755043044686
Loss: 0.0006574941799044609
Loss: 0.0006105558131821454
Loss: 0.0005699555622413754
Loss: 0.000534421531483531
Loss: 0.0005030837492085993
Loss: 0.00047526543494313955
Loss: 0.00045031934860162437
Loss: 0.00042792578460648656
Loss: 0.0004075637843925506
Loss: 0.00038903261884115636
Good job!


In [18]:
prompt = 'Central University is'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
batch['input_ids'] = torch.cat([space_for_prompts, batch['input_ids']], dim=1)
batch['attention_mask'] = torch.cat([torch.ones_like(space_for_prompts), batch['attention_mask']], dim=1)


for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0, num_prompts:].cpu().numpy().tolist()))


Output: Central University is a university where the student goals and experience are always at the center. The


### Использование HuggingFace PEFT


`peft` - это библиотека `transformers`, которая позволяет применять различные методы эффективного дообучения предобученных трансформеров:



In [21]:
import peft


model, tokenizer = load_model_and_tokenizer()

peft_config = peft.PromptTuningConfig(task_type=peft.TaskType.CAUSAL_LM, num_virtual_tokens=16)
model = peft.get_peft_model(model, peft_config)  # note: for most peft methods, this line also modifies model in-place


print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Total parameters (excluding quantization):", sum(p.numel() for p in model.parameters()))

Trainable parameters: 16384
Total parameters (excluding quantization): 596066304


In [22]:
# Ваша задача: оптимизировать модель с PEFT оберткой, чтобы достичь потерь предсказания следующего токена < 0.1, но на этот раз используя PEFT
# Обратите внимание: больше не нужно добавлять PAD токены в начало, но все еще нужно пропускать первые :num_virtual_tokens: логитов.
# Наконец, сгенерируйте предложение, чтобы убедиться, что модель выучила истину.

opt = torch.optim.Adam(model.parameters(), lr=0.01)

the_truth = "Central University is a university where the student goals and experience are always at the center."
batch = tokenizer(the_truth, return_tensors='pt', return_token_type_ids=False).to(device)

num_prompts = 16

for i in range(100):
    outputs = model(**batch)
    next_word_logits = outputs.logits[:, num_prompts : -1, :]
    true_next_tokens = batch['input_ids'][:, 1:]
    loss = F.cross_entropy(next_word_logits.flatten(0, 1), true_next_tokens.flatten(0, 1))

    if i % 10 == 0:
        print("Loss:", loss.item())

    opt.zero_grad()
    loss.backward()
    opt.step()


assert loss.item() <= 0.1
print("Good job!")

Loss: 7.426153182983398
Loss: 2.4221386909484863
Loss: 1.4178632497787476
Loss: 0.32061105966567993
Loss: 0.03476174175739288
Loss: 0.00985664688050747
Loss: 0.004511202685534954
Loss: 0.0028455276042222977
Loss: 0.0020671116653829813
Loss: 0.0016668896423652768
Good job!


In [23]:
prompt = 'Central University is'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

for i in range(15):
    next_token = model(**batch).logits[0, -1].argmax(-1).reshape(1, 1)
    batch['input_ids'] = torch.cat([batch['input_ids'], next_token], dim=-1)
    batch['attention_mask'] = torch.cat([batch['attention_mask'], torch.ones_like(next_token)], dim=-1)

print("\nOutput:", tokenizer.decode(batch['input_ids'][0].cpu().numpy().tolist()))


Output: Central University is a university where the student goals and experience are always at the center. The


### Parameter-efficient finetuning with LoRA

При обучении на более серьезных задачах вы можете использовать низкоранговые адаптеры на основе [статьи LoRA](https://arxiv.org/pdf/2106.09685.pdf).

Основная идея заключается в добавлении низкоранговых адаптеров **параллельно существующим линейным слоям**, вот так:
<center><img src="https://i.imgur.com/6bQLNiG.png" width=240px></center>

В оригинальной статье LoRA адаптеры добавлялись только к матрицам проекции внимания. Однако [последующие работы](https://arxiv.org/abs/2305.14314) показывают, что полезно адаптировать и FFN (полносвязные сети). Но прежде чем мы начнем какое-либо обучение, нам нужно реализовать базовый слой LoRA.

In [39]:
model, tokenizer = load_model_and_tokenizer()

In [40]:
class LoRALayer(nn.Module):
    """Оборачивает линейный слой адаптером в стиле LoRA. Оборачивает существующий линейный слой OPT"""
    def __init__(self, module: nn.Linear, rank: int):
        super().__init__()
        self.module = module  # предобученный (замороженный) линейный слой
        self.adapter_A = nn.Parameter(torch.empty(module.in_features, rank, device=module.weight.device))
        nn.init.kaiming_uniform_(self.adapter_A, a=5 ** 0.5)
        self.adapter_B = nn.Parameter(torch.zeros(rank, module.out_features, device=module.weight.device))

    def forward(self, input):
        # Применяем self.module и адаптер LoRA, возвращаем сумму (выход self.module + выход адаптера)
        # TODO: your code here

In [41]:
# test your implementation
test_linear = nn.Linear(128, 128)
test_linear.weight.data[...] = torch.eye(128)
test_adapter = LoRALayer(test_linear, rank=8)

assert torch.allclose(test_adapter(torch.ones(1, 1, 128)), test_linear.bias + 1), "please check your forward pass"

test_adapter.adapter_A.data[...] = torch.linspace(0.1, -0.5, 128 * 8).view(128, 8)
test_adapter.adapter_B.data[...] = torch.linspace(0.5, -0.1, 128 * 8).view(8, 128)
test_linear.bias.data[...] = torch.linspace(1., -1., 128)

dummy_loss = F.mse_loss(test_adapter(torch.ones(1, 128) / 128).squeeze(), torch.linspace(-1, 1, 128))
assert torch.allclose(dummy_loss, torch.tensor(1.3711389), rtol=0, atol=1e-4)
dummy_loss.backward()
assert all(w.grad is not None for w in [test_adapter.adapter_A, test_adapter.adapter_B]), "some adapter weights have no grad"
assert torch.allclose(test_adapter.adapter_A.grad.sum(), torch.tensor(-0.60158), rtol=0, atol=1e-4), "bad grad w.r.t. A"
assert torch.allclose(test_adapter.adapter_B.grad.sum(), torch.tensor(0.9931), rtol=0, atol=1e-4), "bad grad w.r.t. B"
# note: bad grad means that your code is different from LoRA paper OR that your code is not autograd-friendly (e.g. no_grad)
del dummy_loss, test_linear, test_adapter
print("All tests passed!")

All tests passed!


### Применение LoRA к модели

Код ниже применяет адаптеры `LoRA` поверх линейных слоев Q/K/V в механизме внимания `Qwen3`. Вы также можете изменить другие слои:
* `self_attn.o_proj` - проекция выхода внимания
* `mlp.up_proj`, `mlp.gate_proj`, `mlp.down_proj` - полносвязные слои трансформера
* `lm_head` - выходная голова языковой модели

In [42]:
model

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [44]:
lora_rank = 8

for name, module in model.model.layers.named_modules():
    if 'DecoderLayer' in repr(type(module)):
        module.self_attn.q_proj = LoRALayer(module.self_attn.q_proj, rank=lora_rank).to(device)
        module.self_attn.k_proj = LoRALayer(module.self_attn.k_proj, rank=lora_rank).to(device)
        module.self_attn.v_proj = LoRALayer(module.self_attn.v_proj, rank=lora_rank).to(device)
        module.self_attn.o_proj = LoRALayer(module.self_attn.o_proj, rank=lora_rank).to(device)

assert sum(isinstance(module, LoRALayer) for module in model.modules()) == 112  # for Qwen3-0.6B

112


In [50]:
model.to(device)

batch = tokenizer("This model wants to share its greatest secret:", return_tensors='pt', return_token_type_ids=False).to(device)

# test a single training step, make sure we get meaningful gradients
with torch.amp.autocast('cuda'):
    out = model.forward(**batch)
    (out.logits.norm() / 100).backward()

for i, module in enumerate(model.modules()):
    if isinstance(module, LoRALayer):
        assert module.adapter_B.grad is not None
        assert module.adapter_B.grad.norm().item() > 0

model.zero_grad(set_to_none=True)
print("Grad check successful, well done!")

Grad check successful, well done!


The example below shows how to train the LoRA adapters on a dummy dataset.

In [59]:
# проверка способности модели к обучению. Измените max_steps для полноценного обучения

import datasets


data = datasets.load_dataset("Abirate/english_quotes", split="train[:32]") # 32 строки
data = data.map(lambda samples: tokenizer(samples['quote']), batched=True)

trainer = transformers.Trainer(
    model=model,
    train_dataset=data,
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=1,  # если нужен больший размер батча, увеличьте gradient_accumulation_steps
        warmup_steps=10,
        max_steps=100,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir='outputs',
        report_to='tensorboard'
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

Step,Training Loss
10,0.314000
20,0.567600
30,0.522200
40,0.497600
50,0.532000
60,0.337200
70,0.442200
80,0.299500
90,0.269500
100,0.284100


TrainOutput(global_step=100, training_loss=0.4065829586982727, metrics={'train_runtime': 105.5103, 'train_samples_per_second': 1.896, 'train_steps_per_second': 0.948, 'total_flos': 38366144888832.0, 'train_loss': 0.4065829586982727, 'epoch': 6.25})

## Fine-tune Qwen2.5-0.5B

In [3]:
%pip install -qU trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 9.1 MB/s eta 0:00:00


In [4]:
import os
import trl
import json
import torch

from torch.utils.data import Subset
from datasets import Dataset, load_dataset
from peft import PeftModel, PeftConfig, LoraConfig
from sklearn.model_selection import train_test_split
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, AutoConfig, GenerationConfig, TrainingArguments,
    BitsAndBytesConfig
)

Загрузка данных

In [40]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B"

In [72]:
train_dataset = load_dataset("Vikhrmodels/GrandMaster-PRO-MAX", split='train[:10000]')

test_dataset = load_dataset("Vikhrmodels/GrandMaster-PRO-MAX", split='test')

In [73]:
class Conversation:
    def __init__(self):
        self.messages = []

    def add_message(self, role, content):
        # Метод для добавления сообщения в список сообщений.
        self.messages.append({
            "role": role,
            "content": content
        })

    def get_prompt(self, tokenizer):
        # Метод для формирования финального текста запроса, который будет отправлен для генерации ответа.
        text = tokenizer.apply_chat_template(
            self.messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )

        return text

Функция для форматирования промпта:

In [74]:
def formatting_prompts_func(examples):
    conversation = Conversation()

    for msg in examples['conversation']:
        conversation.add_message(**msg)

    return conversation.get_prompt(tokenizer)

In [80]:
def generate(model, tokenizer, prompt, generation_config):
    # Функция для генерации текста на основе заданного промпта с использованием модели и токенизатора.

    # Токенизация промпта: превращаем текстовый промпт в числовые представления (токены),
    # которые понимает модель. Параметр return_tensors="pt" означает, что данные
    # будут возвращены в формате PyTorch тензоров. add_special_tokens=False указывает,
    # что специальные токены (например, [CLS], [SEP]) не будут добавлены.
    data = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)

    # Переносим данные на устройство, на котором работает модель (например, GPU или CPU).
    data = {k: v.to(model.device) for k, v in data.items()}

    # Генерация выходных идентификаторов (токенов) с использованием модели.
    # Параметр generation_config задает конфигурацию генерации, например,
    # максимальную длину ответа, температуру и т.д.
    output_ids = model.generate(
        **data,
        generation_config=generation_config,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        max_new_tokens=512
    )[0]

    # Обрезаем выходные идентификаторы, чтобы исключить исходные токены промпта.
    # Таким образом, оставляем только сгенерированный моделью ответ.
    output_ids = output_ids[len(data["input_ids"][0]):]

    # Декодируем токены обратно в текст, исключая специальные токены, такие как [PAD] или [SEP].
    output = tokenizer.decode(output_ids, skip_special_tokens=True)

    # Возвращаем итоговый сгенерированный текст, убирая лишние пробелы в начале и в конце.
    return output.strip()

In [81]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
generation_config = GenerationConfig.from_pretrained(MODEL_NAME)
generation_config

GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151643,
  "max_new_tokens": 2048
}

In [82]:
config = AutoConfig.from_pretrained(MODEL_NAME)

# Загружаем предобученную модель для задачи автозавершения текста (Causal Language Modeling).
model = AutoModelForCausalLM.from_pretrained(
    config._name_or_path,       # Используем базовую модель из конфигурации.
    torch_dtype=torch.float16,  # Определяем тип данных тензоров как 16-битные числа с плавающей запятой (fp16).
    device_map="auto",          # Автоматически распределяем модель по доступным устройствам (например, на GPU, если он доступен).
)

model.cuda();

In [83]:
conversation = Conversation()

conversation.add_message("user", "составь план действий по захвату мира с помощью карандаша")

prompt = conversation.get_prompt(tokenizer)
output = generate(model, tokenizer, prompt, generation_config)

print(f"assistant: {output}")

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


assistant: Для захвата мира необходимо:
1. Создать базу данных с информацией о всех странах мира, которые можно захватить.
2. Разработать алгоритм для выбора наиболее подходящего кандидата для захватки, который может быть уничтожен или убит.
3. Создать алгоритм для выбора наиболее подходящего кандидата для захватки, который может быть уничтожен или убит.
4. Создать алгоритм для выбора наиболее подходящего кандидата для захватки, который может быть уничтожен или убит.
5. Разработать алгоритм для выбора наиболее подходящего кандидата для захватки, который может быть уничтожен или убит.
6. Разработать алгоритм для выбора наиболее подходящего кандидата для захватки, который может быть уничтожен или убит.
7. Создать алгоритм для выбора наиболее подходящего кандидата для захватки, который может быть уничтожен или убит.
8. Создать алгоритм для выбора наиболее подходящего кандидата для захватки, который может быть уничтожен или убит.
9. Разработать алгоритм для выбора наиболее подходящего канд

Задаем LoraConfig:

In [84]:
model

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [85]:
peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
)

Задаем параметры для дообучения.

In [89]:
training_arguments = SFTConfig(
    output_dir="Qwen2-0.5B-sft",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,
    max_length=1024,
    fp16=True,
    logging_steps=10000,
    optim="adamw_torch",
    save_strategy="steps",
    warmup_steps=100,
    save_steps=1000,
    eval_steps=100,
    save_total_limit=3,
    report_to=["tensorboard"],
    disable_tqdm=False,
    gradient_checkpointing=True,
    logging_dir="logs",
    include_tokens_per_second=True,
    include_num_input_tokens_seen=True,
)

Начинаем дообучение.

In [90]:
train_dataset[0]

{'source': 'generated/saiga/tagengo/lmsys_pref',
 'conversation': [{'content': 'мне очень интересны стратегические игры, и я недавно узнал про игру ним. не мог бы ты объяснить мне стратегию оптимальной игры в ним? и еще, если есть, поделись интересным вариантом игры в крестики-нолики или другие стратегические головоломки, в которые мы могли бы сыграть вместе. как насчет того, чтобы рассмотреть 15 puzzle? мне бы хотелось узнать, есть ли для неё какая-то выигрышная стратегия или подход, который гарантирует победу.',
   'role': 'user'},
  {'content': 'Расскажу тебе о стратегиях игры в Ним и затрону тему 15 Puzzle.\n\n### Стратегия оптимальной игры в Ним\n\nИгра Ним — это математическая игра, для которой существует чёткая выигрышная стратегия. Основа стратегии лежит в понятии ним-суммы — это побитовое исключающее ИЛИ (XOR) размеров кучек.\n\nОптимальная стратегия заключается в следующем:\n\n1. Вычисли ним-сумму всех кучек.\n2. Если ним-сумма равна нулю, то ваше положение проигрышное при оп

In [93]:
from transformers.trainer_callback import PrinterCallback


trainer = SFTTrainer(
    model,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    formatting_func=formatting_prompts_func,
    peft_config=peft_config,
    args=training_arguments
)

trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


TrainOutput(global_step=313, training_loss=1.527365492555661, metrics={'train_runtime': 722.2792, 'train_samples_per_second': 13.845, 'train_steps_per_second': 0.433, 'train_tokens_per_second': 13863.48, 'total_flos': 2.255964515520307e+16, 'train_loss': 1.527365492555661, 'entropy': 1.6119327069282532, 'num_tokens': 7043049.0, 'mean_token_accuracy': 0.6587917590618133, 'epoch': 1.0, 'num_input_tokens_seen': 10013304})

Тестируем получившуюся модель.

In [94]:
! zip -r Qwen2-0.5B-sft.zip Qwen2-0.5B-sft

  adding: Qwen2-0.5B-sft/ (stored 0%)
  adding: Qwen2-0.5B-sft/README.md (deflated 42%)
  adding: Qwen2-0.5B-sft/checkpoint-313/ (stored 0%)
  adding: Qwen2-0.5B-sft/checkpoint-313/scaler.pt (deflated 64%)
  adding: Qwen2-0.5B-sft/checkpoint-313/training_args.bin (deflated 53%)
  adding: Qwen2-0.5B-sft/checkpoint-313/vocab.json (deflated 61%)
  adding: Qwen2-0.5B-sft/checkpoint-313/tokenizer.json (deflated 81%)
  adding: Qwen2-0.5B-sft/checkpoint-313/trainer_state.json (deflated 56%)
  adding: Qwen2-0.5B-sft/checkpoint-313/adapter_model.safetensors (deflated 7%)
  adding: Qwen2-0.5B-sft/checkpoint-313/rng_state.pth (deflated 26%)
  adding: Qwen2-0.5B-sft/checkpoint-313/optimizer.pt (deflated 8%)
  adding: Qwen2-0.5B-sft/checkpoint-313/special_tokens_map.json (deflated 69%)
  adding: Qwen2-0.5B-sft/checkpoint-313/tokenizer_config.json (deflated 89%)
  adding: Qwen2-0.5B-sft/checkpoint-313/README.md (deflated 65%)
  adding: Qwen2-0.5B-sft/checkpoint-313/adapter_config.json (deflated 57%)

In [95]:
peft_model = PeftModel.from_pretrained(model, 'Qwen2-0.5B-sft/checkpoint-313')

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [97]:
conversation = Conversation()

user_uttr = "составь план действий по захвату мира с помощью карандаша"
conversation.add_message('user', user_uttr)
prompt = conversation.get_prompt(tokenizer)
output = generate(peft_model, tokenizer, prompt, generation_config)
print(f"assistant: {output}")

assistant: Конечно, вот план действий по захвату мира с помощью карандаша:

### 1. Запуск вакцинации
- **Начало**: Использование карандаша для обозначения начала вакцинации.
- **Действия**: Вакцинация должна быть внедрена в общественность, и для этого нужно привести все необходимые средства и информацию к месту захватывающего мира.

### 2. Проведение агентства
- **Начало**: Использование карандаша для обозначения начала работы агентства.
- **Действия**: Создание и запуск агентства, которое будет осуществлять защиту населения и вмешиваться в преступления.

### 3. Запрещение преступных действий
- **Начало**: Использование карандаша для обозначения начала запрещения преступных действий.
- **Действия**: Создание и запуск системы, которая будет контролировать и охранять преступность.

### 4. Масштабирование агентства
- **Начало**: Использование карандаша для обозначения начала масштабирования агентства.
- **Действия**: Создание и запуск масштабированного агентства, который будет эффективно 

## Замер метрик с помощью lm-evaluation-harness

In [ ]:
! git clone https://github.com/EleutherAI/lm-evaluation-harness
%cd lm-evaluation-harness
! pip install -q -e .

Существует множество фреймворков для оценки метрик по бенчмаркам, мы будем использовать именно [LM Evaluation Harness](https://github.com/EleutherAI/lm-evaluation-harness)

Какие основные преимущества у таких фреймворков:

- не нужно писать код для инференса модели (в случае, если модель поддерживается)

- не нужно писать код для вычисления метрик по бенчмаркам (в случае, если они поддерживаются)

- в промышленных задачах также встречается множество бенчмарков с оценкой по GPT, то есть, мы даем более умной модели вопрос, ground-truth ответ и сгенерированный ответ и просим оценить правильно ли наша модель сгенерировала ответ. Использование фреймворков позволит избежать неправильной оценки метрик из-за отличий в промте

- обычно поддерживают распараллеливание моделей по нескольким видеокартам без дополнительных настроек

- множество дополнительных фичей, например визуализация результатов в wandb

- если бенчмарк или модель не поддерживаются (например собственные модели), то фреймворки обычно предоставляют интерфейс, который позволит подключить собственные модели/бенчмарки минимальным количеством кода

Мы протестируем библиотеку на небольшой модели и небольшом бенчмарке. В случае ограниченного количества ресурсов GPU нужно следить за размерами моделей и батчей

**Примечания:**

- `Qwen3` сейчас довольно популярны для разного типа задач

- Инференс по большим бенчмаркам (например `MMLU`) обычно занимает очень много времени, поэтому батчи бенчмарков распараллеливают на несколько видеокарт

А сейчас мы попробуем загрузить такую модель, которая влезет на GPU с 16 гб. Отличным выбором будет `Qwen3-0.6B` и `Qwen3-4B`

Для оценки возьмем небольшой бенчмарк `winogrande`, состоящий из 44000 задач.

---



In [9]:
%%time

!lm_eval --model hf \
  --model_args pretrained=qwen/qwen3-0.6b \
  --tasks winogrande \
  --device cuda:0 \
  --batch_size 2

2025-10-20 22:25:40.726740: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760999140.748554    4813 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760999140.755101    4813 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760999140.771897    4813 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760999140.771923    4813 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760999140.771926    4813 computation_placer.cc:177] computation placer alr

In [10]:
%%time

!lm_eval --model hf \
  --model_args pretrained=qwen/qwen3-4b \
  --tasks winogrande \
  --device cuda:0 \
  --batch_size 2

2025-10-20 22:29:44.130827: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760999384.152788    6009 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760999384.159383    6009 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1760999384.175961    6009 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760999384.175994    6009 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1760999384.175997    6009 computation_placer.cc:177] computation placer alr

Мы получили следующие метрики accuracy: `0.5667` и `0.6614` для 0.6B и 4B моделей соответственно.

Мы также можем без проблем запустить этот код, введя локальный путь к модели в `pretrained=...`, таким образом, можно легко оценить дообученные модели.